# Enhanced Real Estate ML Model Research

Comprehensive machine learning research on Canadian housing data covering:
- **Task 2A: Data Quality & Missing Value Imputation** (Mode, KNN, LDA, Hot Deck, Spatial Interpolation, LOCF)
- **Task 2B: Pretraining Quality & Outlier Treatment** (IQR removal, 1%-99% percentile capping/winsorization, Isolation Forest, Scalers, Log target)
- **Task 2C: Feature Engineering & Analysis** (Domain interactions, metropolitan distance features, polynomial expansions, mutual information ranking)
- **Task 2D: Multi-Model Benchmark & Tuning** (Linear Regression, Ridge, Lasso, ElasticNet, KNN, LDA Price Brackets, Isolation Forest Pipeline, Random Forest, XGBoost, Deep Neural Network MLPRegressor)
- **Model Export**: Exporting trained weights and scaler parameters for browser-side client inference.

In [1]:
# Imports and Path Setup
import sys
if '.' not in sys.path: sys.path.insert(0, '.')
if 'ml_research' not in sys.path: sys.path.insert(0, 'ml_research')
import json
import math
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Load Canadian Housing Dataset
df_raw = pd.read_csv('cleaned_canada.csv')
print(f'Loaded dataset shape: {df_raw.shape}')
print(df_raw.head(3))


Loaded dataset shape: (44896, 23)
           City Province   Latitude   Longitude  ...      Sewer  Pool  Garden  Balcony
0    Revelstoke       BC  50.976585 -118.173149  ...  municipal    No      No       No
1       Boswell       BC  49.471870 -116.770195  ...     septic    No      No       No
2  West Kelowna       BC  49.825230 -119.603253  ...  municipal    No      No       No

[3 rows x 23 columns]


## 1. Task 2A: Data Quality & Missing Data Imputation

We evaluate multiple imputation techniques to address heavy missingness in categorical attributes (Basement, Exterior, Heating, Flooring, Roof):
1. **Mode Baseline**: Mode imputation per column.
2. **KNN Classification**: 5-nearest neighbors in physical feature space.
3. **Linear Discriminant Analysis (LDA)**: Projecting onto discriminant axes.
4. **Hot Deck Imputation**: Finding donor records within the same city and price tier.
5. **Spatial / Linear Interpolation**: Interpolating missing values along coordinate gradients.
6. **LOCF**: Last Observation Carried Forward sequential carry.

In [2]:
from ml_research.data_quality import evaluate_imputation_techniques, impute_dataset

imputation_results = evaluate_imputation_techniques(df_raw, target_col='Basement')
df_imputation_table = pd.DataFrame.from_dict(imputation_results, orient='index')
print(df_imputation_table)

# Impute full dataset using KNN classification
df_imputed = impute_dataset(df_raw, strategy='knn')
print(f'Remaining missing values across dataset: {df_imputed.isna().sum().sum()}')

                                         accuracy  ...                                        description
Mode (Baseline)                          0.757768  ...   Fills missing values with most frequent category
KNN (k=5)                                0.742065  ...  Predicts category using 5 nearest neighbors in...
LDA Imputation                           0.770130  ...  Projected linear discriminant classification f...
Hot Deck                                 0.769796  ...  Matches missing value to donor record in same ...
Linear/Spatial Interpolation             0.706315  ...  Spatial coordinate ordering with nearest gradi...
LOCF (Last Observation Carried Forward)  0.662546  ...        Sequential dataset carry-forward imputation

[6 rows x 3 columns]
Remaining missing values across dataset: 0


## 2. Task 2B: Pretraining Quality & Outlier Treatment

We benchmark pretraining strategies to prevent luxury estate price distortions while retaining representative market variance:
- **IQR Outlier Removal (m=3.0 vs m=1.5)**: Eliminates extreme records.
- **Winsorization / Percentile Capping (1% - 99%)**: Retains 100% of data points by clipping extreme values to boundaries.
- **Isolation Forest**: Unsupervised multivariate anomaly filtering.
- **RobustScaler**: Scaling based on median and interquartile range.

In [3]:
from ml_research.pretraining_quality import evaluate_pretraining_treatments, cap_outliers_percentile

pretrain_bench = evaluate_pretraining_treatments(
    df_imputed,
    feature_cols=['Bedrooms', 'Bathrooms', 'Square Footage', 'Latitude', 'Longitude'],
    target_col='Price',
    outlier_cols=['Price', 'Square Footage']
)
df_pretrain_table = pd.DataFrame.from_dict(pretrain_bench, orient='index')
print(df_pretrain_table)

# Apply 1%-99% percentile capping to preserve full dataset
df_clean = cap_outliers_percentile(df_imputed, ['Price', 'Square Footage'], lower_pct=0.01, upper_pct=0.99)
print(f'Preserved dataset size: {len(df_clean)} records')

                                               samples  ...          rmse
1. Baseline (Raw + StandardScale)                44896  ...  1.580960e+06
2. IQR Removal (m=3.0 + StandardScale)           42973  ...  4.341380e+05
3. IQR Removal (m=1.5 + StandardScale)           40222  ...  3.182844e+05
4. Percentile Capping (1%-99% + RobustScale)     44896  ...  6.848562e+05
5. Isolation Forest (c=0.02 + RobustScale)       43998  ...  5.729265e+05
6. Capped (1%-99%) + Log(Price) + RobustScale    44896  ...  8.554419e+05

[6 rows x 6 columns]
Preserved dataset size: 44896 records


## 3. Task 2C: Feature Engineering & Selection Analysis

We design domain-specific features and systematically evaluate their contribution:
- **Interactions & Ratios**: sqft_per_bedroom, sqft_per_room, bath_bed_ratio, sqft_x_bedrooms, sqft_x_bathrooms, bed_bath_product, luxury_score, amenity_density.
- **Geographic Metro Proximity**: Euclidean distance to Vancouver, Toronto, Montreal, Calgary, Ottawa, and min_dist_to_metro.
- **Polynomial Features**: Quadratic expansions sqft_squared, bedrooms_squared, bathrooms_squared.
- **Mutual Information Ranking**: Identifying the top predictive features.

In [4]:
from ml_research.feature_engineering import add_engineered_features, compare_feature_subsets, analyze_feature_importance

df_feat, feature_groups = add_engineered_features(df_clean)
subset_comparison = compare_feature_subsets(df_feat, feature_groups, target_col='Price')
print(subset_comparison)

top_features = analyze_feature_importance(df_feat, feature_groups['full_model'][:25], target_col='Price')
print(top_features.head(15))

    Feature_Set  Num_Features  R2_Score           RMSE
0    full_model            48  0.665950  618418.472818
1      baseline             6  0.590374  684810.090253
2  interactions             8  0.487757  765798.878955
3    polynomial             3  0.460366  786006.926813
4   categorical            18  0.336052  871853.716073
5      location             8  0.235923  935287.460228
6     amenities             7  0.136751  994133.271507
              Feature  Mutual_Information  Pearson_Correlation
0   dist_to_vancouver            0.414766            -0.331976
1     dist_to_toronto            0.410647             0.318918
2           Longitude            0.403022            -0.326317
3    dist_to_montreal            0.390633             0.332133
4   min_dist_to_metro            0.386759            -0.384012
5    sqft_x_bathrooms            0.342482             0.639507
6            Latitude            0.331662            -0.035771
7      Square Footage            0.310719             0.

## 4. Task 2D: Multi-Model Training & Benchmark Leaderboard

We train and compare 10 diverse regression models with hyperparameter tuning:
1. Linear Regression (OLS)
2. Ridge Regression (L2 regularized, alpha=10.0)
3. Lasso Regression (L1 regularized, alpha=0.01)
4. ElasticNet (alpha=0.1, l1_ratio=0.5)
5. K-Nearest Neighbors Regressor (k=7, distance-weighted)
6. Random Forest Regressor (n_estimators=100, max_depth=16)
7. XGBoost Regressor (n_estimators=150, max_depth=6, lr=0.08)
8. Deep Neural Network (MLPRegressor 128x64)
9. Isolation Forest + Ridge Pipeline
10. LDA Price Bracket Classifier + Ridge

In [5]:
from ml_research.model_training import train_and_evaluate_models, tune_winning_models, export_model_for_browser

features = feature_groups['full_model']
X = df_feat[features].values
y = df_feat['Price'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

leaderboard, fitted_models = train_and_evaluate_models(
    X_train_scaled, X_test_scaled, y_train, y_test, features
)
print("=== ALGORITHM BENCHMARK LEADERBOARD ===")
print(leaderboard)

print("=== HYPERPARAMETER TUNING FOR WINNING MODELS ===")
tuning_results = tune_winning_models(X_train_scaled, X_test_scaled, y_train, y_test)
for model_name, configs in tuning_results.items():
    print(f"{model_name} Sweep: {configs}")

# Export dual Random Forest + Ridge model for in-browser client inference
from sklearn.ensemble import RandomForestRegressor
rf_browser = RandomForestRegressor(n_estimators=40, max_depth=10, random_state=42, n_jobs=-1)
rf_browser.fit(X_train_scaled, y_train)
ridge_model = fitted_models["2. Ridge Regression (alpha=10.0)"]
best_metrics = {
    "random_forest_r2": round(float(r2_score(y_test, rf_browser.predict(X_test_scaled))), 4),
    "ridge_r2": leaderboard[leaderboard["Model"].str.contains("Ridge")]["R2_Score"].iloc[0],
}
export_model_for_browser(
    model=rf_browser,
    scaler=scaler,
    feature_names=features,
    metrics=best_metrics,
    export_path="ml_research/exported_models/best_model_meta.json",
    ridge_model=ridge_model,
)
print("Exported dual browser inference model successfully!")


=== ALGORITHM BENCHMARK LEADERBOARD ===
                                     Model  R2_Score  ...       RMSE  Fit_Time_Sec
0   6. Random Forest (n=100, max_depth=16)    0.8790  ...  372120.10         11.10
1  7. XGBoost Regressor (depth=6, lr=0.08)    0.8783  ...  373214.89          0.48
2         5. KNN Regressor (k=7, distance)    0.7147  ...  571493.26          0.00
3      8. Deep Neural Network (MLP 128x64)    0.7126  ...  573568.81        100.01
4            10. LDA Price Bracket + Ridge    0.6794  ...  605821.96          0.60
5               1. Linear Regression (OLS)    0.6660  ...  618364.71          0.23
6         3. Lasso Regression (alpha=0.01)    0.6652  ...  619118.36          5.01
7         2. Ridge Regression (alpha=10.0)    0.6639  ...  620345.65          0.02
8        4. ElasticNet (alpha=0.1, l1=0.5)    0.6375  ...  644184.13          2.69
9     9. Isolation Forest + Ridge Pipeline    0.5787  ...  694490.16          0.50

[10 rows x 5 columns]
=== HYPERPARAMETER TUNIN

/home/agy/.local/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.139e+15, tolerance: 4.175e+12
  model = cd_fast.enet_coordinate_descent(
/home/agy/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
